### Setup

In [54]:
# !pip install librecommender faker tensorflow torch

In [55]:
# !pip install tensorflow==2.15.0 keras==2.15.0 librecommender==1.5.2

In [56]:
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from collections import defaultdict
from faker import Faker
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from libreco.data import DatasetPure
from libreco.algorithms import UserCF, ItemCF, ALS, BPR
from libreco.evaluation import evaluate

In [57]:
spark = (SparkSession.builder
    .appName("JobRecommender")
    .config("spark.sql.catalog.nessie.ref", "test")
    .getOrCreate()
)

In [58]:
jobs_df = spark.read.format("iceberg").load("nessie.silver.jobs")
jobs_df.show(5)

+--------------------+--------------+------------+--------------------+--------------+--------------------+----------+----------+--------+-----------+---------+---------+---------------+------------------+--------------+------------------+-------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+
|             job_key|processed_date|    platform|         title_clean|location_clean|       company_clean|min_salary|max_salary|currency|salary_type|min_years|max_years|experience_type|education_standard|level_standard|work_form_standard|quantity_normalized|expired_date_norm|         tech_skills|         soft_skills|          skills_all| category_name_final|         description|         requirement|                link|gold_processed|
+--------------------+--------------+------------+--------------------+--------------+--------------------+----------+--

### Lấy dữ liệu việc làm mới nhất

In [59]:
w = Window.partitionBy(
    "job_key"
).orderBy(
    F.col("processed_date").desc()
)

latest_jobs = (
    jobs_df
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)

In [60]:
jobs_df = (
    latest_jobs
    .select(
        "job_key",
        "location_clean",
        "min_salary",
        "max_salary",
        "min_years",
        "max_years",
        "work_form_standard",
        "category_name_final",
        "skills_all"
    )
).toPandas()

jobs_df.head()

,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all
0,00036788d38de1c9d1433e9f36e8ba0de40233bf0ce39c...,Hà Nội,15000000.0,30000000.0,2.0,2.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[React Next JS, Node JS, React JS Java Script,..."
1,001aa442de3e83d685c55fad31227c883c098c0729395f...,Hà Nội,8000000.0,NaN,0.0,0.0,full_time,HOẠT ĐỘNG DỊCH VỤ KHÁC,"[Word, Excel, giao tiếp tiếng Trung, giao tiếp]"
2,002862efca2eef130d31db889706d06c4ae73a2e501642...,Đà Nẵng,50000000.0,NaN,0.0,0.0,full_time,HOẠT ĐỘNG KINH DOANH BẤT ĐỘNG SẢN,[Giao tiếp]
3,003c504f4f710bba48f2b05c5d98971057ddfd7745e04d...,Hồ Chí Minh,NaN,NaN,0.0,1.0,full_time,XÂY DỰNG,"[Auto CAD, triển khai bản vẽ shop drawing CAD ..."
4,003d99eae190f5178c817588b18b722adfbd9dc0aed4cc...,Hà Nội,15000000.0,30000000.0,0.0,0.0,full_time,GIÁO DỤC VÀ ĐÀO TẠO,"[giao tiếp, thuyết phục, chốt sale, Làm việc n..."


### Mapping skill


In [61]:
skill_map_df = pd.read_csv("notebooks/skill_freq_6k.csv", on_bad_lines='skip')

skill_lookup = {
    row["raw_skill_norm"].strip().lower(): row["canonical"].strip()
    for _, row in skill_map_df.iterrows()
    if pd.notna(row["canonical"])
}

In [62]:
import re

def normalize_skill(s: str) -> str | None:
    """Chuẩn hóa 1 skill string, trả về canonical hoặc None nếu không có trong dict."""
    s = str(s).strip().lower()
    s = re.sub(r'\s+', ' ', s)  # collapse whitespace
    return skill_lookup.get(s)  # None nếu không match

def normalize_skills_list(skills) -> list:
    """Normalize toàn bộ list skills của 1 job, bỏ skill không có trong dict."""
    if not hasattr(skills, '__iter__'):
        return []
    result = []
    seen = set()
    for s in skills:
        canonical = normalize_skill(s)
        if canonical and canonical not in seen:
            seen.add(canonical)
            result.append(canonical)
    return result

In [63]:
# Thay thế skills_all bằng version đã normalize
jobs_df["skills_all"] = jobs_df["skills_all"].apply(normalize_skills_list)

# Chỉ giữ job có ít nhất 1 skill sau normalize
jobs_df = jobs_df[jobs_df["skills_all"].apply(len) > 0].copy()

# Dùng skills_norm thay cho skills_all ở mọi chỗ
jobs_by_category = defaultdict(list)
for _, job in jobs_df.iterrows():
    jobs_by_category[job["category_name_final"]].append(job)

### Sinh dữ liệu tương tác người dùng

#### Gom job theo category

In [17]:
jobs_by_category = defaultdict(list)

for _, job in jobs_df.iterrows():
    jobs_by_category[
        job["category_name_final"]
    ].append(job)

#### Sinh user

In [18]:

# =========================
# 1. Clean jobs
# =========================

def clean_skill_list(skills):
    if not isinstance(skills, list):
        return []

    cleaned = []

    for s in skills:
        s = str(s).strip().lower()

        if not s:
            continue

        cleaned.append(s)

    return list(dict.fromkeys(cleaned))


jobs_df = jobs_df.copy()

jobs_df["skills_clean"] = jobs_df["skills_all"].apply(
    clean_skill_list
)

jobs_df = jobs_df[
    jobs_df["skills_clean"].apply(lambda x: len(x) > 0)
].copy()

jobs_df = jobs_df.dropna(
    subset=["category_name_final"]
)

# =========================
# 2. Build category pools
# =========================

jobs_by_category = defaultdict(list)

for _, job in jobs_df.iterrows():
    jobs_by_category[job["category_name_final"]].append(job)

MIN_JOBS_PER_CATEGORY = 300

valid_categories = [
    cate
    for cate, rows in jobs_by_category.items()
    if len(rows) >= MIN_JOBS_PER_CATEGORY
]

print("Valid categories:", len(valid_categories))

# =========================
# 3. Balanced user categories
# =========================

N_USERS = 10000

users_per_category = N_USERS // len(valid_categories)

user_categories = []

for cate in valid_categories:
    user_categories.extend(
        [cate] * users_per_category
    )

while len(user_categories) < N_USERS:
    user_categories.append(
        random.choice(valid_categories)
    )

random.shuffle(user_categories)

# =========================
# 4. Salary fallback stats
# =========================

category_salary_stats = (
    jobs_df
    .groupby("category_name_final")
    .agg({
        "min_salary": "median",
        "max_salary": "median"
    })
)

global_min_salary = jobs_df["min_salary"].median()
global_max_salary = jobs_df["max_salary"].median()

if pd.isna(global_min_salary):
    global_min_salary = 10_000_000

if pd.isna(global_max_salary):
    global_max_salary = 25_000_000

# =========================
# 5. Helper functions
# =========================

def safe_salary(category, col, default_value):
    value = np.nan

    if category in category_salary_stats.index:
        value = category_salary_stats.loc[
            category,
            col
        ]

    if pd.isna(value):
        value = default_value

    return value


def safe_exp(value, default_value):
    if pd.isna(value):
        return default_value

    return value


# =========================
# 6. Generate users
# =========================

users = []

for i in range(N_USERS):

    category = user_categories[i]
    category_jobs = jobs_by_category[category]

    seed_count = min(
        random.randint(2, 3),
        len(category_jobs)
    )

    seed_jobs = random.sample(
        category_jobs,
        seed_count
    )

    primary_job = random.choice(seed_jobs)

    # -----------------------
    # skill profile
    # -----------------------

    primary_skills = (
        primary_job["skills_clean"]
        if isinstance(primary_job["skills_clean"], list)
        else []
    )

    related_skill_pool = set(primary_skills)

    for job in seed_jobs:
        skills = job["skills_clean"]

        if isinstance(skills, list):
            related_skill_pool.update(skills)

    related_skill_pool = list(related_skill_pool)

    if len(primary_skills) > 0:

        min_base = max(
            1,
            int(len(primary_skills) * 0.6)
        )

        max_base = len(primary_skills)

        n_base = random.randint(
            min_base,
            max_base
        )

        base_skills = random.sample(
            primary_skills,
            n_base
        )

    else:
        base_skills = []

    extra_pool = list(
        set(related_skill_pool) - set(base_skills)
    )

    n_extra = random.randint(
        0,
        min(3, len(extra_pool))
    )

    extra_skills = random.sample(
        extra_pool,
        n_extra
    )

    user_skills = list(
        dict.fromkeys(base_skills + extra_skills)
    )

    # fallback nếu vẫn quá ít skill
    if len(user_skills) < 2 and len(related_skill_pool) > 0:
        user_skills = random.sample(
            related_skill_pool,
            min(3, len(related_skill_pool))
        )

    # -----------------------
    # experience profile
    # -----------------------

    min_exp = safe_exp(
        primary_job["min_years"],
        0
    )

    max_exp = safe_exp(
        primary_job["max_years"],
        min_exp + 2
    )

    lower = max(
        0,
        int(min_exp) - 1
    )

    upper = max(
        lower + 1,
        int(max_exp) + 1
    )

    years_exp = random.randint(
        lower,
        upper
    )

    # -----------------------
    # salary profile
    # -----------------------

    base_min_salary = safe_salary(
        category,
        "min_salary",
        global_min_salary
    )

    base_max_salary = safe_salary(
        category,
        "max_salary",
        global_max_salary
    )

    if base_max_salary < base_min_salary:
        base_max_salary = base_min_salary * 1.5

    desired_min_salary = int(
        base_min_salary *
        random.uniform(0.8, 1.0)
    )

    desired_max_salary = int(
        base_max_salary *
        random.uniform(1.0, 1.2)
    )

    if desired_max_salary <= desired_min_salary:
        desired_max_salary = int(
            desired_min_salary *
            random.uniform(1.2, 1.6)
        )

    # -----------------------
    # location
    # -----------------------

    locations = [
        job["location_clean"]
        for job in seed_jobs
        if pd.notna(job["location_clean"])
    ]

    location = (
        random.choice(locations)
        if locations
        else "Không xác định"
    )

    # -----------------------
    # work form
    # -----------------------

    work_forms = [
        job["work_form_standard"]
        for job in seed_jobs
        if pd.notna(job["work_form_standard"])
    ]

    preferred_work_form = (
        random.choice(work_forms)
        if work_forms
        else "unknown"
    )

    users.append({
        "user_id": f"U{i:05d}",
        "location": location,
        "preferred_category": category,
        "years_experience": years_exp,
        "desired_min_salary": desired_min_salary,
        "desired_max_salary": desired_max_salary,
        "preferred_work_form": preferred_work_form,
        "skills": user_skills,
        "seed_job_key": primary_job["job_key"]
    })

users_df = pd.DataFrame(users)

users_df.head()

Valid categories: 15


,user_id,location,preferred_category,years_experience,desired_min_salary,desired_max_salary,preferred_work_form,skills,seed_job_key
0,U00000,Bắc Ninh,HOẠT ĐỘNG DỊCH VỤ KHÁC,1,8289610,15534465,full_time,"[đàm phán, tiếp thị, lập kế hoạch, đề xuất giả...",074b4f8c9eebbc343e2891c580d87f95ceed0176f640a1...
1,U00001,Thái Bình,"VẬN TẢI, KHO BÃI",2,10316772,17582304,full_time,"[xuất nhập khẩu, __skip__, microsoft office, l...",a39d62ffcaf9d7a91ff109b47b430ec14fdb38a9ae42eb...
2,U00002,Hồ Chí Minh,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",1,11696117,21973211,full_time,"[network, ielts, làm việc nhóm, tiếng anh, __s...",50989d948fec44c342e94ea2837d9a0035a6d51531f80b...
3,U00003,Hà Nội,"HOẠT ĐỘNG XUẤT BẢN, PHÁT SÓNG, SẢN XUẤT VÀ PHÂ...",3,9976976,17837321,full_time,"[phân tích dữ liệu, youtube, social media, tik...",c4424b485fff62ce39d4f87ae952a268eeaa263aa740a8...
4,U00004,Hồ Chí Minh,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",2,11531580,22038783,full_time,"[python, lập trình, tiếng anh, tensorflow, cấu...",a36b6766b84d90c5b196f75f779823cec7e9f0276fb18f...


Sinh user sao cho rải đều các category

In [19]:
users_df["preferred_category"].value_counts()

preferred_category
HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ                                                               669
HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ VẤN, CƠ SỞ HẠ TẦNG MÁY TÍNH VÀ CÁC DỊCH VỤ THÔNG TIN KHÁC    667
HOẠT ĐỘNG DỊCH VỤ KHÁC                                                                                    667
BÁN BUÔN VÀ BÁN LẺ                                                                                        667
HOẠT ĐỘNG XUẤT BẢN, PHÁT SÓNG, SẢN XUẤT VÀ PHÂN PHỐI NỘI DUNG                                             667
CÔNG NGHIỆP CHẾ BIẾN, CHẾ TẠO                                                                             667
Y TẾ VÀ HOẠT ĐỘNG TRỢ GIÚP XÃ HỘI                                                                         667
XÂY DỰNG                                                                                                  667
VẬN TẢI, KHO BÃI                                                                                     

In [20]:
users_spark = spark.createDataFrame(
    users_df
)

users_spark.write.format("iceberg").mode("overwrite").saveAsTable("nessie.silver.users")

In [21]:
users_spark.show(5)

+-------+-----------+--------------------+----------------+------------------+------------------+-------------------+--------------------+--------------------+
|user_id|   location|  preferred_category|years_experience|desired_min_salary|desired_max_salary|preferred_work_form|              skills|        seed_job_key|
+-------+-----------+--------------------+----------------+------------------+------------------+-------------------+--------------------+--------------------+
| U00000|   Bắc Ninh|HOẠT ĐỘNG DỊCH VỤ...|               1|           8289610|          15534465|          full_time|[đàm phán, tiếp t...|074b4f8c9eebbc343...|
| U00001|  Thái Bình|    VẬN TẢI, KHO BÃI|               2|          10316772|          17582304|          full_time|[xuất nhập khẩu, ...|a39d62ffcaf9d7a91...|
| U00002|Hồ Chí Minh|HOẠT ĐỘNG VIỄN TH...|               1|          11696117|          21973211|          full_time|[network, ielts, ...|50989d948fec44c34...|
| U00003|     Hà Nội|HOẠT ĐỘNG XUẤT BẢ..

#### Sinh interaction

In [22]:
def skill_match_ratio(
    user_skills,
    job_skills
):

    if not user_skills or not job_skills:
        return 0

    user_skills = set(user_skills)
    job_skills = set(job_skills)

    overlap = len(
        user_skills &
        job_skills
    )

    return overlap / len(job_skills)

In [23]:
def calculate_match_score(
    user,
    job
):

    score = 0

    # 0 -> 10
    skill_ratio = skill_match_ratio(
        user["skills"],
        job["skills_all"]
    )

    score += skill_ratio * 10

    # same category
    if (
        user["preferred_category"]
        ==
        job["category_name_final"]
    ):
        score += 3

    # work form
    if (
        user["preferred_work_form"]
        ==
        job["work_form_standard"]
    ):
        score += 1

    # location
    if (
        user["location"]
        ==
        job["location_clean"]
    ):
        score += 1

    # experience
    min_years = (
        job["min_years"]
        if pd.notna(job["min_years"])
        else 0
    )

    max_years = (
        job["max_years"]
        if pd.notna(job["max_years"])
        else min_years + 2
    )

    if (
        min_years
        <=
        user["years_experience"]
        <=
        max_years
    ):
        score += 2

    return score

In [24]:
interactions = []

job_records = jobs_df.to_dict("records")

jobs_by_category = defaultdict(list)

for job in job_records:
    jobs_by_category[job["category_name_final"]].append(job)

rating_map = {
    "view": 1,
    "save": 3,
    "apply": 5
}

for idx, user in users_df.iterrows():
    user_category = user["preferred_category"]
    same_category_jobs = jobs_by_category.get(user_category, [])

    sampled_jobs = []

    # Tăng pool lên để có nhiều candidate hơn
    sampled_jobs.extend(
        random.sample(same_category_jobs, min(80, len(same_category_jobs)))  # 45 → 80
    )

    random_jobs = random.sample(job_records, min(50, len(job_records)))  # 30 → 50
    related_other_jobs = []

    for job in random_jobs:
        if job["category_name_final"] == user_category:
            continue
        skill_ratio = skill_match_ratio(user["skills"], job["skills_all"])  # fix: dùng skills_norm
        if skill_ratio > 0:
            related_other_jobs.append(job)

    sampled_jobs.extend(
        random.sample(related_other_jobs, min(10, len(related_other_jobs)))  # 5 → 10
    )

    sampled_jobs = list({job["job_key"]: job for job in sampled_jobs}.values())

    for job in sampled_jobs:
        score = calculate_match_score(user, job)

        if score < 2:  # 5 → 2, bớt loại sớm
            continue

        if job["category_name_final"] != user_category:
            ratio = skill_match_ratio(user["skills"], job["skills_all"])  # fix: skills_all
            if ratio < 0.15:  # 0.3 → 0.15
                continue

        # Tăng probability để tạo nhiều interaction hơn
        probability = min(score / 10, 0.92)  # 16 → 10

        if random.random() > probability:
            continue

        # Giữ nguyên event weights
        if score >= 12:
            event = random.choices(["view", "save", "apply"], weights=[15, 30, 55])[0]
        elif score >= 9:
            event = random.choices(["view", "save", "apply"], weights=[45, 45, 10])[0]
        elif score >= 5:
            event = random.choices(["view", "save"], weights=[85, 15])[0]
        else:
            event = "view"  # score 2-4: chỉ view

        interactions.append({
            "user_id": user["user_id"],
            "job_key": job["job_key"],
            "event_type": event,
            "rating": rating_map[event],
            "event_time": datetime.now() - timedelta(
                days=random.randint(0, 180),
                hours=random.randint(0, 23),
                minutes=random.randint(0, 59)
            )
        })

interactions_df = pd.DataFrame(interactions)

print(interactions_df.shape)
interactions_df["event_type"].value_counts(normalize=True)

(378909, 5)


event_type
view    0.918247
save    0.081753
Name: proportion, dtype: float64

In [25]:
interaction_spark = spark.createDataFrame(
    interactions_df
)

interaction_spark.write.format("iceberg").mode("overwrite").saveAsTable("nessie.silver.interactions")

In [26]:
interaction_spark.show(5)

+-------+--------------------+----------+------+--------------------+
|user_id|             job_key|event_type|rating|          event_time|
+-------+--------------------+----------+------+--------------------+
| U00000|5b946af6c7f812061...|      view|     1|2026-05-03 17:06:...|
| U00000|e84c7a85aabdeb431...|      view|     1|2026-06-10 16:57:...|
| U00000|3c806cab0ebc524f0...|      view|     1|2026-06-10 14:11:...|
| U00000|375e363d7bed60311...|      view|     1|2026-05-02 18:17:...|
| U00000|12940ee2adabe29b6...|      view|     1|2026-02-21 06:09:...|
+-------+--------------------+----------+------+--------------------+
only showing top 5 rows



### (Optional) Lấy dữ liệu từ bảng nếu đã sinh dữ liệu

In [11]:
users_df = spark.read.format("iceberg").load("nessie.silver.users").toPandas()
interactions_df = spark.read.format("iceberg").load("nessie.silver.interactions").toPandas()

### (Optional) Lấy dữ liệu từ ở ngoài

In [ ]:
# Load users and interactions from csv files (if not using Iceberg)
users_df = pd.read_csv("notebooks/users.csv")
interactions_df = pd.read_csv("notebooks/interactions.csv")

In [ ]:
import json

users_df["skills"] = users_df["skills"].apply(
    json.loads
)

### Tiền xử lý interactions

In [64]:
interaction_cf = (
    interactions_df
    .groupby(["user_id", "job_key"])["rating"]
    .max()
    .reset_index()
)

interaction_cf = interaction_cf.rename(
    columns={
        "user_id": "user",
        "job_key": "item",
        "rating": "label"
    }
)

interaction_cf.head()

,user,item,label
0,U00000,02f5411a351c0063c05a8c26ff87de28c881f9a813fd51...,1
1,U00000,11b844f1e6d1bf1fc9517125193589201c4c7b4c051962...,1
2,U00000,12940ee2adabe29b69865b07b9f483ae29774080a94dee...,1
3,U00000,1795a15cf41df37c334b43bf804f2878989b82e2456b29...,1
4,U00000,20de8f55b7ba35f8ba3266e56a45e656ab0a35e309d70a...,1


### Tạo train/test

In [65]:
from sklearn.model_selection import train_test_split

train_parts = []
test_parts = []

for user, group in interaction_cf.groupby("user"):

    if len(group) < 2:
        train_parts.append(group)
        continue

    train_g, test_g = train_test_split(
        group,
        test_size=0.2,
        random_state=42
    )

    train_parts.append(train_g)
    test_parts.append(test_g)

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

test_df = test_df[
    test_df["item"].isin(train_df["item"])
].copy()

In [66]:
train_data, data_info = DatasetPure.build_trainset(
    train_df
)

test_data = DatasetPure.build_testset(
    test_df,
    data_info
)

In [78]:
data_info.save(
    path="./notebooks/models",
    model_name="data_info"
)

#### Tạo tập train / test riêng cho BPR

In [67]:
ranking_df = interaction_cf.copy()
ranking_df["label"] = 1

train_rank_df, test_rank_df = train_test_split(
    ranking_df,
    test_size=0.2,
    random_state=42
)

train_rank_data, rank_data_info = DatasetPure.build_trainset(
    train_rank_df
)

test_rank_data = DatasetPure.build_testset(
    test_rank_df,
    rank_data_info
)

### Train mô hình

#### UserCF

In [68]:
usercf = UserCF(
    task="rating",
    data_info=data_info,
    sim_type="cosine",
    k_sim=100
)

usercf.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-13 10:41:04
Final block size and num: (10000, 1)
sim_matrix elapsed: 0.992s
sim_matrix, shape: (10000, 10000), num_elements: 2420832, density: 2.4208 %


top_k: 100%|██████████| 10000/10000 [00:00<00:00, 16566.65it/s]

In [ ]:
# Xuất mô hình
usercf.save(model_name='usercf_model', path="./notebooks/models")

file folder ./models doesn't exists, creating a new one...


#### ItemCF

In [ ]:
itemcf = ItemCF(
    task="rating",
    data_info=data_info,
    sim_type="cosine",
    k_sim=100
)

itemcf.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-11 12:22:36
Final block size and num: (3093, 20)
sim_matrix elapsed: 17.920s
sim_matrix, shape: (61842, 61842), num_elements: 7251866, density: 0.1896 %


top_k: 100%|██████████| 61842/61842 [00:03<00:00, 18133.16it/s]

#### ALS

In [ ]:
als = ALS(
    task="rating",
    data_info=data_info,
    embed_size=32,
    n_epochs=20,
    reg=1e-4,
    seed=42
)

als.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-11 12:22:58
Epoch 1 elapsed: 0.376s
Epoch 2 elapsed: 0.254s
Epoch 3 elapsed: 0.259s
Epoch 4 elapsed: 0.261s
Epoch 5 elapsed: 0.258s
Epoch 6 elapsed: 0.255s
Epoch 7 elapsed: 0.256s
Epoch 8 elapsed: 0.285s
Epoch 9 elapsed: 0.334s
Epoch 10 elapsed: 0.254s
Epoch 11 elapsed: 0.243s
Epoch 12 elapsed: 0.284s
Epoch 13 elapsed: 0.263s
Epoch 14 elapsed: 0.249s
Epoch 15 elapsed: 0.249s
Epoch 16 elapsed: 0.244s
Epoch 17 elapsed: 0.251s
Epoch 18 elapsed: 0.249s
Epoch 19 elapsed: 0.250s
Epoch 20 elapsed: 0.251s


#### BPR

In [ ]:
bpr = BPR(
    task="ranking",
    data_info=rank_data_info,
    embed_size=32,
    n_epochs=20,
    lr=0.001,
    reg=1e-4,
    seed=42
)

bpr.fit(
    train_rank_data,
    neg_sampling=True,
    verbose=2
)

Training start time: 2026-06-11 12:23:03


train: 100%|██████████| 1185/1185 [00:15<00:00, 75.22it/s]


Epoch 1 elapsed: 15.756s
	 train_loss: 0.6834


train: 100%|██████████| 1185/1185 [00:14<00:00, 81.55it/s]


Epoch 2 elapsed: 14.537s
	 train_loss: 0.6694


...

Epoch 18 elapsed: 11.167s
	 train_loss: 0.6575


train: 100%|██████████| 1185/1185 [00:10<00:00, 116.77it/s]


Epoch 19 elapsed: 10.150s
	 train_loss: 0.6577


train: 100%|██████████| 1185/1185 [00:10<00:00, 113.91it/s]

Epoch 20 elapsed: 10.405s
	 train_loss: 0.6576


### Recommend cho 1 user

Chọn user U00123

In [36]:
user_id = "U00004"
user_idx = 4

In [37]:
users_df.loc[user_idx]

user_id                                                           U00004
location                                                     Hồ Chí Minh
preferred_category     HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...
years_experience                                                       2
desired_min_salary                                              11531580
desired_max_salary                                              22038783
preferred_work_form                                            full_time
skills                 [python, lập trình, tiếng anh, tensorflow, cấu...
seed_job_key           a36b6766b84d90c5b196f75f779823cec7e9f0276fb18f...
Name: 4, dtype: object

In [38]:
for skill in users_df.loc[user_idx, "skills"]:
    print(skill)

python
lập trình
tiếng anh
tensorflow
cấu trúc dữ liệu
java
pytorch
toeic
scikit-learn
agile
automation


#### Thử nghiệm trên 4 mô hình

In [39]:
user_recs = usercf.recommend_user(
    user=user_id,
    n_rec=20
)

user_res = user_recs[user_id]
enriched_user = jobs_df[
    jobs_df["job_key"].isin(user_res)
]
enriched_user['method'] = 'user'
enriched_user.head()

/tmp/ipykernel_81178/850884489.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_user['method'] = 'user'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,method
7633,51afddad21d083bb59678f8cd0ffdd9ffe171d1b224e88...,Hồ Chí Minh,1.0,2.0,2.0,4.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[An toàn lao động, __SKIP__, TypeScript, LangC...",user
9570,e6a4ab6cd4f290ff8b0f0946a878d7bdd383a10c220189...,Hà Nội,8000000.0,15000000.0,1.0,1.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[Testing, Business Analysis, ChatGPT, Sáng tạo...",user
10047,0ba6efdeb314f330b9fccd2ac036108c423328c498b5e2...,Hà Nội,20000000.0,20000000.0,2.0,2.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[Testing, Tiếng Anh, Giải quyết vấn đề]",user
14466,608ae43f3e81109134fe4e47f65e2565fed4963ba93469...,Hồ Chí Minh,NaN,NaN,2.0,2.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[__SKIP__, SQL Server, Debugging, .NET, Entity...",user
18569,a013848b8e12e0c9c605f49feb15e6bc500d381c3ef78c...,Hồ Chí Minh,NaN,NaN,2.0,2.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[Altium, Product Design]",user


In [40]:
item_recs = itemcf.recommend_user( 
    user=user_id,
    n_rec=20
)

item_res = item_recs[user_id]
enriched_item = jobs_df[
    jobs_df["job_key"].isin(item_res)
]
enriched_item['method'] = 'item'
enriched_item.head()

/tmp/ipykernel_81178/3104140752.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_item['method'] = 'item'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,method
6235,e3e6fed8dceb0e4e5724f8a9c787b7bf60e34097fed66a...,Hồ Chí Minh,15000000.0,20000000.0,1.0,1.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[System design, __SKIP__, Network, VMware, Bán...",item
7633,51afddad21d083bb59678f8cd0ffdd9ffe171d1b224e88...,Hồ Chí Minh,1.0,2.0,2.0,4.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[An toàn lao động, __SKIP__, TypeScript, LangC...",item
9570,e6a4ab6cd4f290ff8b0f0946a878d7bdd383a10c220189...,Hà Nội,8000000.0,15000000.0,1.0,1.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[Testing, Business Analysis, ChatGPT, Sáng tạo...",item
9828,fa351fcc784afb5277f476b2107070e4fafc956af17e11...,Hồ Chí Minh,15000000.0,20000000.0,1.0,1.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[System design, __SKIP__, Network, VMware, Bán...",item
10047,0ba6efdeb314f330b9fccd2ac036108c423328c498b5e2...,Hà Nội,20000000.0,20000000.0,2.0,2.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[Testing, Tiếng Anh, Giải quyết vấn đề]",item


In [41]:
als_recs = als.recommend_user(
    user=user_id,   
    n_rec=20    
)

als_res = als_recs[user_id]
enriched_als = jobs_df[
    jobs_df["job_key"].isin(als_res)    
]
enriched_als['method'] = 'als'
enriched_als.head()

/tmp/ipykernel_81178/1723471370.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_als['method'] = 'als'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,method
1243,6244a8c0b28798c4a69faf29f743cfd8dab091e7241209...,Hưng Yên,NaN,NaN,3.0,0.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[Kaizen, ISO 14001, IATF, __SKIP__, Quản lý]",als
4429,5c8c062cb3fd31c1a66b22d888a67f320faf2243663a82...,Hồ Chí Minh,NaN,NaN,1.0,1.0,full_time,"HOẠT ĐỘNG XUẤT BẢN, PHÁT SÓNG, SẢN XUẤT VÀ PHÂ...","[Marketing, __SKIP__, Tiếng Anh, Microsoft Off...",als
6050,d732f9b9000340b0993244c8de7c826eb0b8579e3fbfad...,Hà Nội,8000000.0,12000000.0,0.0,1.0,full_time,"NGHỆ THUẬT, THỂ THAO VÀ GIẢI TRÍ","[CorelDRAW, Photoshop, Quản lý thời gian, Làm ...",als
11207,65976343f6f7642acfd28095e486a8684fb923e5b2c237...,Hà Nội,15000000.0,18000000.0,0.0,0.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[Tiếng Trung, Tiếng Anh, Thuyết trình, Chăm só...",als
11808,9345a3a9eca5f985c6008725e78ac295adaa4e1cac8438...,Hồ Chí Minh,17000000.0,20000000.0,3.0,4.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[Tư vấn tuyển sinh, Marketing, Phân tích dữ li...",als


In [42]:
bpr_recs = bpr.recommend_user(
    user=user_id,
    n_rec=20
)

bpr_res = bpr_recs[user_id]
enriched_bpr = jobs_df[
    jobs_df["job_key"].isin(bpr_res)
]
enriched_bpr['method'] = 'bpr'
enriched_bpr.head()

/tmp/ipykernel_81178/2677664316.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_bpr['method'] = 'bpr'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,method
809,3fda073194c71065aa4e10824af334eb844e7ab547905c...,Hồ Chí Minh,NaN,NaN,1.0,3.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[__SKIP__, Ra quyết định, Làm việc nhóm]",bpr
1748,878bb2bfc18ed298d005cd2674e9c3ac3beb8645844748...,Hồ Chí Minh,NaN,NaN,2.0,3.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[Tiếng Anh, Bán hàng]",bpr
4390,58d0877e245c1f0876ba8827c4018548460cab245c094a...,Hồ Chí Minh,NaN,NaN,2.0,5.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[Tiếng Trung, hàn điện, Tiếng Nhật, Tiếng Anh,...",bpr
4480,5ff83a0ae457fea413ff6afa480538ab6a017533dc12f4...,Hồ Chí Minh,NaN,NaN,1.0,1.0,full_time,"VẬN TẢI, KHO BÃI","[QA/QC, Quản lý chất lượng, Quy trình vận hành...",bpr
13773,2acded1abc262f2350c9f9f4acdfc37d2842514491de48...,Hồ Chí Minh,NaN,NaN,0.0,0.0,full_time,"VẬN TẢI, KHO BÃI",[Tiếng Anh],bpr


### Đánh giá mô hình

#### BPR

In [27]:


evaluate(
    model=bpr,
    data=test_rank_data,
    metrics=[
        "precision",
        "recall",
        "ndcg"
    ],
    neg_sampling=True,
    k=10
)

eval_listwise:  10%|▉         | 966/9999 [00:14<02:32, 59.40it/s]

eval_listwise: 100%|██████████| 9999/9999 [02:20<00:00, 71.16it/s] 


{'precision': 0.0019101910191019103,
 'recall': 0.002614136427517766,
 'ndcg': 0.008196019929446676}

#### UserCF

In [28]:
evaluate(
    model=usercf,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise:   0%|          | 0/10 [00:00<?, ?it/s]

No common interaction or similar neighbor for user 4113 and item 22342, proceed with default prediction
No common interaction or similar neighbor for user 3488 and item 12408, proceed with default prediction
No common interaction or similar neighbor for user 2857 and item 45593, proceed with default prediction
No common interaction or similar neighbor for user 4752 and item 48941, proceed with default prediction
No common interaction or similar neighbor for user 7416 and item 21412, proceed with default prediction
No common interaction or similar neighbor for user 6339 and item 44603, proceed with default prediction


eval_pointwise: 100%|██████████| 10/10 [00:04<00:00,  2.22it/s]


{'rmse': 0.6307335829835442, 'mae': 0.2926323053956448}

#### ItemCF

In [29]:
evaluate(
    model=itemcf,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise:   0%|          | 0/10 [00:00<?, ?it/s]

No common interaction or similar neighbor for user 4113 and item 22342, proceed with default prediction
No common interaction or similar neighbor for user 2857 and item 45593, proceed with default prediction
No common interaction or similar neighbor for user 4752 and item 48941, proceed with default prediction
No common interaction or similar neighbor for user 7416 and item 21412, proceed with default prediction
No common interaction or similar neighbor for user 6339 and item 44603, proceed with default prediction
No common interaction or similar neighbor for user 8503 and item 41653, proceed with default prediction


eval_pointwise: 100%|██████████| 10/10 [00:04<00:00,  2.26it/s]


{'rmse': 0.6269529066510553, 'mae': 0.29351053776592134}

#### ALS

In [30]:
evaluate(
    model=als,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise: 100%|██████████| 10/10 [00:00<00:00, 463.89it/s]


{'rmse': 0.6452408, 'mae': 0.26355788}

### Đánh giá chất lượng đề xuất

#### Avg Skill Overlap

In [51]:
def clean_skill_list(skills):
    if not isinstance(skills, list):
        return []

    cleaned = []

    for s in skills:
        s = str(s).strip().lower()

        if not s:
            continue

        cleaned.append(s)

    return list(dict.fromkeys(cleaned))

def avg_skill_overlap(
    recommended_jobs,
    user_skills
):

    overlaps = []

    # clean skills all with clean_skill_list
    recommended_jobs["skills_clean"] = [clean_skill_list(skills) for skills in recommended_jobs["skills_all"]]

    for skills in recommended_jobs["skills_clean"]:

        overlap = len(
            set(user_skills)
            &
            set(skills)
        )

        overlaps.append(overlap)

    return np.mean(overlaps)

In [52]:
print ("UserCF skill overlap:",
    avg_skill_overlap(
        enriched_user,
        users_df.loc[user_idx, "skills"]
    )
)

print ("ItemCF skill overlap:",
    avg_skill_overlap(
        enriched_item,
        users_df.loc[user_idx, "skills"]
    )
)

print ("ALS skill overlap:",
    avg_skill_overlap(
        enriched_als,
        users_df.loc[user_idx, "skills"]
    )
)

print ("BPR skill overlap:",
    avg_skill_overlap(
        enriched_bpr,
        users_df.loc[user_idx, "skills"]
    )
)

UserCF skill overlap: 1.05
ItemCF skill overlap: 1.05
ALS skill overlap: 0.45
BPR skill overlap: 0.75


/tmp/ipykernel_81178/2222622434.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recommended_jobs["skills_clean"] = [clean_skill_list(skills) for skills in recommended_jobs["skills_all"]]
/tmp/ipykernel_81178/2222622434.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recommended_jobs["skills_clean"] = [clean_skill_list(skills) for skills in recommended_jobs["skills_all"]]
/tmp/ipykernel_81178/2222622434.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram

#### Avg Category Match

In [45]:
def avg_category_match(
    recommended_jobs,
    user_category
):

    return (
        recommended_jobs[
            "category_name_final"
        ]
        ==
        user_category
    ).mean()

In [53]:
print ("UserCF category m:",
    avg_category_match(
        enriched_user,
        users_df.loc[user_idx, "preferred_category"]
    )
)

print ("ItemCF category m:",
    avg_category_match(
        enriched_item,
        users_df.loc[user_idx, "preferred_category"]
    )
)

print ("ALS category m:",
    avg_category_match(
        enriched_als,
        users_df.loc[user_idx, "preferred_category"]
    )
)

print ("BPR category m:",
    avg_category_match(
        enriched_bpr,
        users_df.loc[user_idx, "preferred_category"]
    )
)

UserCF category m: 1.0
ItemCF category m: 1.0
ALS category m: 0.05
BPR category m: 0.0


#### Avg Experience Match

In [47]:
def avg_exp_match(
    recommended_jobs,
    years_exp
):

    matches = []

    for _, row in recommended_jobs.iterrows():

        min_exp = (
            row["min_years"]
            if pd.notna(row["min_years"])
            else 0
        )

        max_exp = (
            row["max_years"]
            if pd.notna(row["max_years"])
            else 100
        )

        matches.append(
            min_exp <= years_exp <= max_exp
        )

    return np.mean(matches)

In [48]:
print("UserCF exp match:",
    avg_exp_match(
        enriched_user,
        users_df.loc[user_idx, "years_experience"]
    )
)

print("ItemCF exp match:",
    avg_exp_match(
        enriched_item,
        users_df.loc[user_idx, "years_experience"]
    )
)   

print("ALS exp match:",
    avg_exp_match(
        enriched_als,
        users_df.loc[user_idx, "years_experience"]
    )
)

print("BPR exp match:",
    avg_exp_match(
        enriched_bpr,
        users_df.loc[user_idx, "years_experience"]
    )
)

UserCF exp match: 0.4
ItemCF exp match: 0.2
ALS exp match: 0.15
BPR exp match: 0.6


### (Optional) Checking

In [54]:
check_df = (
    interactions_df
    .merge(
        users_df[["user_id", "preferred_category", "skills"]],
        on="user_id",
        how="left"
    )
    .merge(
        jobs_df[["job_key", "category_name_final", "skills_all"]],
        on="job_key",
        how="left"
    )
)

In [55]:
check_df["same_category"] = (
    check_df["preferred_category"]
    ==
    check_df["category_name_final"]
)

check_df["same_category"].mean()

0.9999947216877931

In [56]:
def skill_overlap_count(user_skills, job_skills):
    if not isinstance(user_skills, list) or not isinstance(job_skills, list):
        return 0

    user_set = set(
        str(s).strip().lower()
        for s in user_skills
    )

    job_set = set(
        str(s).strip().lower()
        for s in job_skills
    )

    return len(user_set & job_set)

In [57]:
check_df["skill_overlap"] = check_df.apply(
    lambda row: skill_overlap_count(
        row["skills"],
        row["skills_all"]
    ),
    axis=1
)

In [58]:
(check_df["skill_overlap"] > 0).mean()

0.5845044588542367

In [59]:
check_df.groupby("event_type")["skill_overlap"].mean()

event_type
save    0.96152
view    0.92510
Name: skill_overlap, dtype: float64

In [60]:
bad_cases = check_df[
    (check_df["same_category"] == False)
    &
    (check_df["skill_overlap"] == 0)
]

bad_cases[
    [
        "user_id",
        "preferred_category",
        "category_name_final",
        "skills",
        "skills_all",
        "event_type",
        "rating"
    ]
].head(20)

,user_id,preferred_category,category_name_final,skills,skills_all,event_type,rating


In [61]:
summary = pd.DataFrame({
    "same_category_rate": [check_df["same_category"].mean()],
    "has_skill_overlap_rate": [(check_df["skill_overlap"] > 0).mean()],
    "avg_skill_overlap": [check_df["skill_overlap"].mean()]
})

summary

,same_category_rate,has_skill_overlap_rate,avg_skill_overlap
0,0.999995,0.584504,0.928078


In [62]:
users_df[
    users_df["skills"].apply(
        lambda skills:
            any(
                "python" in str(s).lower()
                for s in skills
            )
            if isinstance(skills, list)
            else False
    )
]

,user_id,location,preferred_category,years_experience,desired_min_salary,desired_max_salary,preferred_work_form,skills,seed_job_key
4,U00004,Hồ Chí Minh,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",2,11531580,22038783,full_time,"[python, lập trình, tiếng anh, tensorflow, cấu...",a36b6766b84d90c5b196f75f779823cec7e9f0276fb18f...
57,U00057,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",1,12408608,23335870,full_time,"[python, tiếng anh, thuyết phục, pytorch, giao...",a59138a3e37e9d33ea0c35f3765717f12f2f9b8de5f8d9...
186,U00186,Khác,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",4,12360888,20913205,full_time,"[giao tiếp, tiếng anh, power bi, python, phân ...",d596d94f0a29d8f821dad3eaababd8341b243533de2e11...
272,U00272,Hồ Chí Minh,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",2,11696591,23692663,full_time,"[python, data science, computer science, softw...",aceba6cae3dac2f90e42f95f70159a7c69cca3967f12ef...
429,U00429,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",1,13736141,20062629,full_time,"[v-ray, unreal engine, photoshop, sáng tạo, da...",5e71aca5711a1e695c4307d152d2a6ddaba38d4d8214ea...
...,...,...,...,...,...,...,...,...,...
9669,U09669,Hồ Chí Minh,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",3,13789557,23026608,full_time,"[ai, documentation, data visualization, rag, d...",77e868157506cf46b5753fd1bc44916a9dee743813b2cc...
9709,U09709,Hồ Chí Minh,Y TẾ VÀ HOẠT ĐỘNG TRỢ GIÚP XÃ HỘI,5,11820466,20886667,full_time,"[làm việc nhóm, __skip__, power bi, python, mi...",7e8889c8ec996dc96bb1dc7308613b52e82ca75d973a5c...
9760,U09760,Hồ Chí Minh,GIÁO DỤC VÀ ĐÀO TẠO,2,11240845,22369233,part_time,"[robotics, lập trình, scratch, giảng dạy, unit...",284bbaf8384ae4f0f9222a6823df838e6ade10d1085469...
9892,U09892,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",2,12533053,20069827,full_time,"[system design, optimization, go (golang), pyt...",ad3c8df28de2f27a66962136987026834cca0d47f298db...


In [63]:
def precision_recall_at_k(model, test_df, train_df, k=10):
    results = []
    
    for user in test_df["user"].unique():
        # Ground truth: items user tương tác trong test
        ground_truth = set(test_df[test_df["user"] == user]["item"])
        
        # Items đã thấy trong train (loại khỏi recommend)
        seen = set(train_df[train_df["user"] == user]["item"])
        
        # Get top-K recommendations
        try:
            recs = model.recommend_user(user=user, n_rec=k)
            rec_items = set(recs[user])
        except:
            continue
        
        hits = len(ground_truth & rec_items)
        precision = hits / k
        recall = hits / len(ground_truth) if ground_truth else 0
        
        results.append({"precision": precision, "recall": recall})
    
    df = pd.DataFrame(results)
    return df.mean()

In [64]:
print("UserCF Precision@10, Recall@10:",
    precision_recall_at_k(
        usercf,
        test_df,
        train_df,
        k=10
    )
)

print("ItemCF Precision@10, Recall@10:",
    precision_recall_at_k(
        itemcf,
        test_df,
        train_df,
        k=10
    )
)

print("ALS Precision@10, Recall@10:",
    precision_recall_at_k(
        als,
        test_df,
        train_df,
        k=10
    )
)

UserCF Precision@10, Recall@10: precision    0.006680
recall       0.008725
dtype: float64
ItemCF Precision@10, Recall@10: precision    0.006570
recall       0.008513
dtype: float64
ALS Precision@10, Recall@10: precision    0.000260
recall       0.000322
dtype: float64


### (Test) Train mô hình

#### UserCF

In [1]:
usercf = UserCF(
    task="ranking",
    data_info=rank_data_info,
    sim_type="cosine",
    k_sim=100
)

usercf.fit(
    train_data,
    neg_sampling=True,
    verbose=2
)

NameError: name 'UserCF' is not defined

#### ItemCF

In [ ]:
itemcf = ItemCF(
    task="rating",
    data_info=rank_data_info,
    sim_type="cosine",
    k_sim=100
)

itemcf.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-11 09:37:05
Final block size and num: (3093, 20)
sim_matrix elapsed: 20.717s
sim_matrix, shape: (61842, 61842), num_elements: 7251866, density: 0.1896 %


top_k: 100%|██████████| 61842/61842 [00:03<00:00, 15759.48it/s]

#### ALS

In [ ]:
als = ALS(
    task="rating",
    data_info=rank_data_info,
    embed_size=32,
    n_epochs=20,
    reg=1e-4,
    seed=42
)

als.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-11 09:37:30
Epoch 1 elapsed: 0.291s
Epoch 2 elapsed: 0.282s
Epoch 3 elapsed: 0.406s
Epoch 4 elapsed: 0.270s
Epoch 5 elapsed: 0.272s
Epoch 6 elapsed: 0.258s
Epoch 7 elapsed: 0.264s
Epoch 8 elapsed: 0.268s
Epoch 9 elapsed: 0.265s
Epoch 10 elapsed: 0.263s
Epoch 11 elapsed: 0.286s
Epoch 12 elapsed: 0.285s
Epoch 13 elapsed: 0.249s
Epoch 14 elapsed: 0.265s
Epoch 15 elapsed: 0.281s
Epoch 16 elapsed: 0.265s
Epoch 17 elapsed: 0.270s
Epoch 18 elapsed: 0.271s
Epoch 19 elapsed: 0.269s
Epoch 20 elapsed: 0.261s


In [ ]:
bpr = BPR(
    task="ranking",
    data_info=rank_data_info,
    embed_size=32,
    n_epochs=20,
    lr=0.001,
    reg=1e-4,
    seed=42
)

bpr.fit(
    train_rank_data,
    neg_sampling=True,
    verbose=2
)